# 🧠 TARS HELIX LITE — Обучение

| Уровень | Время (A100) | Steps | Описание |
|---------|-------------|-------|----------|
| `small` | ~10 мин | 200 | Тест что всё работает |
| `medium` | ~2 часа | 1500 | Стандарт |
| `max` | ~10 часов | 10000 | Продакшн |
| `marathon` | ~4 дня | 75000 | Полный |

**Порядок:** сначала `small` → если ОК → переключи на `medium`/`max` и запусти заново.

In [ ]:
#@title ⚙️ Настройки {display-mode: "form"}
#@markdown ---
#@markdown ### Уровень обучения
LEVEL = 'small'  #@param ['small', 'medium', 'max', 'marathon']
#@markdown ### Продолжить с чекпоинта?
RESUME = True  #@param {type:"boolean"}
#@markdown ---

In [ ]:
#@title 1. 💾 Google Drive + GPU + Проект
from google.colab import drive
drive.mount('/content/drive')

import torch, os, shutil
assert torch.cuda.is_available(), '❌ GPU не найден! Runtime → Change runtime type → A100/T4'
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
bf16 = torch.cuda.is_bf16_supported()
print(f'✅ GPU: {gpu} ({vram:.0f} GB, bf16={bf16})')

# Клонировать проект
P = '/content/TarsSSM-Py'
DRIVE_P = '/content/drive/MyDrive/TARS/TarsSSM-Py'

if os.path.exists(P):
    print('📁 Проект уже есть, обновляю...')
    !cd {P} && git pull --ff-only 2>/dev/null || true
elif os.path.exists(DRIVE_P):
    print('📁 Копирую с Google Drive...')
    !cp -r {DRIVE_P} {P}
else:
    print('📥 Клонирую с GitHub...')
    !git clone https://github.com/Vazilll/TarsSSM-Py.git {P}

os.chdir(P)

# Зависимости
!pip install -q einops tqdm datasets 2>/dev/null

# Восстановить чекпоинты с Drive если есть
DRIVE_CKPT = '/content/drive/MyDrive/TarsData/models/tars_lite'
LOCAL_CKPT = '/content/TarsSSM-Py/models/tars_lite'
if os.path.exists(DRIVE_CKPT):
    os.makedirs(LOCAL_CKPT, exist_ok=True)
    for f in os.listdir(DRIVE_CKPT):
        if f.endswith('.pt'):
            src = os.path.join(DRIVE_CKPT, f)
            dst = os.path.join(LOCAL_CKPT, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f'  📥 {f} ({os.path.getsize(src)/1e6:.0f} MB)')

print(f'\n📂 {os.getcwd()}')
print(f'🎯 Уровень: {LEVEL}')

In [ ]:
#@title 2. 🧪 Smoke Test
!python local_train.py --test-only

In [ ]:
#@title 3. 🚀 ОБУЧЕНИЕ
# local_train.py сам:
#   - определит GPU и выберет оптимальные batch/d_model/n_layers
#   - скачает данные из HuggingFace если нет локальных
#   - применит WSD schedule + Muon optimizer
#   - сохранит чекпоинты в models/tars_lite/
#   - скопирует на Google Drive

resume_flag = '--resume' if RESUME else ''

!python local_train.py --level {LEVEL} --drive colab --no-update {resume_flag}

In [ ]:
#@title 4. 💾 Финальное сохранение на Drive
import shutil, os

# local_train.py уже сохраняет на Drive, но на всякий случай
DRIVE_CKPT = '/content/drive/MyDrive/TarsData/models/tars_lite'
LOCAL_CKPT = '/content/TarsSSM-Py/models/tars_lite'
os.makedirs(DRIVE_CKPT, exist_ok=True)

if os.path.exists(LOCAL_CKPT):
    for f in os.listdir(LOCAL_CKPT):
        if f.endswith('.pt'):
            src = os.path.join(LOCAL_CKPT, f)
            dst = os.path.join(DRIVE_CKPT, f)
            shutil.copy2(src, dst)
            print(f'💾 {f} → Drive ({os.path.getsize(src)/1e6:.0f} MB)')

# Сохранить проект на Drive (первый раз)
DRIVE_P = '/content/drive/MyDrive/TARS/TarsSSM-Py'
if not os.path.exists(DRIVE_P):
    print('\n📦 Сохраняю проект на Drive...')
    shutil.copytree('/content/TarsSSM-Py', DRIVE_P,
                    ignore=shutil.ignore_patterns('__pycache__', '.git', 'venv', 'node_modules', '*.pyc'))

# state файл
state_src = '/content/TarsSSM-Py/train_state.json'
if os.path.exists(state_src):
    shutil.copy2(state_src, os.path.join(DRIVE_CKPT, 'train_state.json'))

print(f'\n✅ Всё на Drive: {DRIVE_CKPT}')
!ls -lh {DRIVE_CKPT}/ 2>/dev/null

In [ ]:
#@title 5. 🔍 Проверка модели
import torch, sys, os
sys.path.insert(0, '/content/TarsSSM-Py')
from config import TarsConfig
from brain.mamba2.core.model_lite import TarsHelixLite

# Найти лучший checkpoint
paths = [
    '/content/TarsSSM-Py/models/tars_lite/checkpoint_best.pt',
    '/content/drive/MyDrive/TarsData/models/tars_lite/checkpoint_best.pt',
]
ckpt_path = next((p for p in paths if os.path.exists(p)), None)

if ckpt_path:
    ckpt = torch.load(ckpt_path, map_location='cuda', weights_only=False)
    cfg_d = ckpt.get('config', {})
    cfg = TarsConfig(**{k:v for k,v in cfg_d.items() if hasattr(TarsConfig,k)})
    model = TarsHelixLite(cfg).cuda()
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.eval()
    
    n = sum(p.numel() for p in model.parameters())
    print(f'✅ Модель: {n/1e6:.1f}M params (d={cfg.d_model}, L={cfg.n_layers})')
    print(f'   Epoch: {ckpt.get("epoch","?")}, Loss: {ckpt.get("loss", ckpt.get("best_loss","?"))}')
    
    # Генерация
    prompt = torch.randint(0, 256, (1, 16), device='cuda')
    with torch.no_grad():
        out = model.generate(prompt, max_new_tokens=64, temperature=0.8)
    raw = bytes(out[0].cpu().tolist())
    text = raw.decode('utf-8', errors='replace')
    print(f'\n💬 Генерация ({out.shape[1]} tokens):')
    print(f'   {text[:200]}')
else:
    print('❌ Checkpoint не найден — сначала запусти обучение!')